# Fine-tuning e regiões utilizadas por uma CNN

Notebook independente para o artigo. Ele não usa `experiment.py` nem outros arquivos do projeto: baixa o Oxford-IIIT Pet, treina a ResNet-18, calcula as métricas e gera os mapas Grad-CAM.

Execute as células em ordem. A etapa principal é controlada por `RUN_FULL`; deixe-a como `False` na primeira execução e use o teste rápido antes.

In [ ]:
!pip -q install 'torch>=2.2' 'torchvision>=0.17' matplotlib pandas pillow

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv, json, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch import nn
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import InterpolationMode

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('/content/cnn_data')
RESULTS_DIR = Path('/content/cnn_results')
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
STRATEGIES = ('frozen', 'full')
print('Dispositivo:', DEVICE)

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def stratified_split(labels, validation_fraction=0.20, train_fraction=1.0, seed=42):
    grouped = defaultdict(list)
    for index, label in enumerate(labels): grouped[int(label)].append(index)
    rng = random.Random(seed); train_indices=[]; validation_indices=[]
    for indices in grouped.values():
        rng.shuffle(indices); n_val=max(1,round(len(indices)*validation_fraction))
        validation_indices.extend(indices[:n_val]); candidates=indices[n_val:]
        train_indices.extend(candidates[:max(1,round(len(candidates)*train_fraction))])
    rng.shuffle(train_indices); rng.shuffle(validation_indices)
    return train_indices, validation_indices

class PetEvaluationDataset(Dataset):
    def __init__(self, root):
        self.dataset=OxfordIIITPet(root=root,split='test',target_types=('category','segmentation'),download=True)
        self.image_transform=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
        self.mask_transform=transforms.Compose([transforms.Resize(256,interpolation=InterpolationMode.NEAREST),transforms.CenterCrop(224),transforms.PILToTensor()])
    def __len__(self): return len(self.dataset)
    def __getitem__(self,index):
        image,target=self.dataset[index]; label,trimap=target; mask=self.mask_transform(trimap).squeeze(0)
        return self.image_transform(image),int(label),(mask!=2)

def make_loaders(seed=42,train_fraction=1.0,test_limit=0,batch_size=64,workers=0):
    train_transform=transforms.Compose([transforms.RandomResizedCrop(224,scale=(0.75,1.0)),transforms.RandomHorizontalFlip(),transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.10),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    eval_transform=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    metadata=OxfordIIITPet(root=DATA_DIR,split='trainval',target_types='category',download=True)
    labels=getattr(metadata,'_labels',None)
    if labels is None: labels=[metadata[i][1] for i in range(len(metadata))]
    train_indices,val_indices=stratified_split(labels,seed=seed,train_fraction=train_fraction)
    train_data=OxfordIIITPet(root=DATA_DIR,split='trainval',target_types='category',transform=train_transform,download=False)
    val_data=OxfordIIITPet(root=DATA_DIR,split='trainval',target_types='category',transform=eval_transform,download=False)
    test_data=PetEvaluationDataset(DATA_DIR)
    if test_limit: test_data=Subset(test_data,range(min(test_limit,len(test_data))))
    options=dict(batch_size=batch_size,num_workers=workers,pin_memory=torch.cuda.is_available())
    generator=torch.Generator().manual_seed(seed)
    train_loader=DataLoader(Subset(train_data,train_indices),shuffle=True,generator=generator,**options)
    val_loader=DataLoader(Subset(val_data,val_indices),shuffle=False,**options)
    test_loader=DataLoader(test_data,shuffle=False,**options)
    return train_loader,val_loader,test_loader,metadata.classes


In [ ]:
def build_model(strategy,n_classes):
    model=resnet18(weights=ResNet18_Weights.DEFAULT); model.fc=nn.Linear(model.fc.in_features,n_classes)
    if strategy=='frozen':
        for parameter in model.parameters(): parameter.requires_grad=False
        for parameter in model.fc.parameters(): parameter.requires_grad=True
    return model.to(DEVICE)

def make_optimizer(model,strategy,lr_head=1e-3,lr_backbone=1e-4):
    if strategy=='frozen': return torch.optim.AdamW(model.fc.parameters(),lr=lr_head,weight_decay=1e-4)
    backbone=[p for name,p in model.named_parameters() if not name.startswith('fc.')]
    return torch.optim.AdamW([{'params':backbone,'lr':lr_backbone},{'params':model.fc.parameters(),'lr':lr_head}],weight_decay=1e-4)

def classification_metrics(labels,predictions,probabilities,n_classes):
    encoded=labels*n_classes+predictions; matrix=torch.bincount(encoded,minlength=n_classes*n_classes).reshape(n_classes,n_classes).double(); tp=matrix.diag()
    precision=tp/matrix.sum(0).clamp_min(1); recall=tp/matrix.sum(1).clamp_min(1); f1=2*precision*recall/(precision+recall).clamp_min(1e-12)
    confidence=probabilities.max(1).values; correct=predictions.eq(labels).float(); ece=torch.tensor(0.0)
    edges=torch.linspace(0,1,11)
    for low,high in zip(edges[:-1],edges[1:]):
        selected=(confidence>low)&(confidence<=high)
        if selected.any(): ece += selected.float().mean()*(correct[selected].mean()-confidence[selected].mean()).abs()
    return {'accuracy':float(tp.sum()/matrix.sum().clamp_min(1)),'macro_f1':float(f1.mean()),'balanced_accuracy':float(recall.mean()),'ece_10_bins':float(ece)}

@torch.no_grad()
def evaluate(model,loader,n_classes):
    model.eval(); labels_all=[]; predictions_all=[]; probabilities_all=[]; loss_sum=0; n=0
    for batch in loader:
        images,labels=batch[0].to(DEVICE),batch[1].to(DEVICE); logits=model(images); probabilities=logits.softmax(1); predictions=probabilities.argmax(1)
        loss_sum+=float(F.cross_entropy(logits,labels))*images.size(0); n+=images.size(0); labels_all.append(labels.cpu()); predictions_all.append(predictions.cpu()); probabilities_all.append(probabilities.cpu())
    labels=torch.cat(labels_all); predictions=torch.cat(predictions_all); probabilities=torch.cat(probabilities_all); metrics=classification_metrics(labels,predictions,probabilities,n_classes); metrics['loss']=loss_sum/max(n,1); return metrics

def train(model,strategy,train_loader,val_loader,n_classes,epochs,outdir):
    optimizer=make_optimizer(model,strategy); scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs); scaler=GradScaler('cuda',enabled=DEVICE.type=='cuda'); criterion=nn.CrossEntropyLoss(label_smoothing=.1); best=-1; history=[]
    for epoch in range(1,epochs+1):
        model.train()
        if strategy=='frozen': model.eval(); model.fc.train()
        total=count=0
        for images,labels in train_loader:
            images,labels=images.to(DEVICE),labels.to(DEVICE); optimizer.zero_grad(set_to_none=True)
            with autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'): loss=criterion(model(images),labels)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); total+=float(loss.detach())*images.size(0); count+=images.size(0)
        scheduler.step(); val=evaluate(model,val_loader,n_classes); record={'epoch':epoch,'train_loss':total/max(count,1),**{'validation_'+k:v for k,v in val.items()}}; history.append(record)
        print(f'{strategy} | época {epoch}/{epochs} | loss={record["train_loss"]:.4f} | macro-F1={val["macro_f1"]:.4f}')
        if val['macro_f1']>best: best=val['macro_f1']; torch.save(model.state_dict(),outdir/f'resnet18_{strategy}.pt')
    model.load_state_dict(torch.load(outdir/f'resnet18_{strategy}.pt',map_location=DEVICE,weights_only=True)); (outdir/f'history_{strategy}.json').write_text(json.dumps(history,indent=2),encoding='utf-8'); return model


In [ ]:
class GradCAM:
    def __init__(self,model,layer): self.model=model; self.activations=None; self.gradients=None; self.handle=layer.register_forward_hook(self.forward_hook)
    def forward_hook(self,module,inputs,output):
        self.activations=output
        if output.requires_grad: output.register_hook(lambda gradient:setattr(self,'gradients',gradient))
    def __call__(self,image):
        self.model.zero_grad(set_to_none=True); image=image.detach().clone().requires_grad_(True); logits=self.model(image); prediction=int(logits.argmax(1).item()); logits[0,prediction].backward()
        weights=self.gradients.mean((2,3),keepdim=True); cam=torch.relu((weights*self.activations).sum(1,keepdim=True)); cam=F.interpolate(cam,size=image.shape[-2:],mode='bilinear',align_corners=False)[0,0]; cam=(cam-cam.min())/(cam.max()-cam.min()).clamp_min(1e-12); return logits.detach(),prediction,cam.detach().cpu()
    def close(self): self.handle.remove()

def spatial_metrics(cam,foreground,quantile=.80):
    salient=cam>=torch.quantile(cam.flatten(),quantile); foreground=foreground.bool(); intersection=(salient&foreground).sum().float(); union=(salient|foreground).sum().float().clamp_min(1)
    return {'foreground_energy':float(cam[foreground].sum()/cam.sum().clamp_min(1e-12)),'pointing_game':float(foreground.flatten()[int(cam.argmax())]),'foreground_saliency_iou':float(intersection/union)}

def map_similarity(first,second,quantile=.80):
    a,b=first.flatten().float(),second.flatten().float(); cosine=float(F.cosine_similarity(a,b,dim=0)); ma,mb=a>=torch.quantile(a,quantile),b>=torch.quantile(b,quantile); return cosine,float((ma&mb).sum().float()/(ma|mb).sum().float().clamp_min(1))

def unnormalize(image):
    mean=torch.tensor(MEAN).view(3,1,1); std=torch.tensor(STD).view(3,1,1); return ((image.cpu()*std+mean).clamp(0,1)).permute(1,2,0).numpy()

def analyze_gradcam(models,test_loader,class_names,outdir,max_samples=200):
    explainers={s:GradCAM(models[s],models[s].layer4[-1].conv2) for s in STRATEGIES}; rows=[]; paired=[]; examples=[]; processed=0
    for images,labels,masks in test_loader:
        for index in range(images.size(0)):
            if processed>=max_samples: break
            image=images[index:index+1].to(DEVICE); label=int(labels[index]); results={}; sample={'image':images[index],'label':label}
            for strategy in STRATEGIES:
                logits,prediction,cam=explainers[strategy](image); metrics=spatial_metrics(cam,masks[index]); confidence=float(logits.softmax(1)[0,prediction]); results[strategy]={'prediction':prediction,'cam':cam,**metrics}; rows.append({'sample':processed,'strategy':strategy,'label':label,'prediction':prediction,'correct':int(prediction==label),'confidence':confidence,**metrics})
            cosine,overlap=map_similarity(results['frozen']['cam'],results['full']['cam']); paired.append({'sample':processed,'same_prediction':int(results['frozen']['prediction']==results['full']['prediction']),'foreground_energy_delta':results['full']['foreground_energy']-results['frozen']['foreground_energy'],'pointing_game_delta':results['full']['pointing_game']-results['frozen']['pointing_game'],'foreground_saliency_iou_delta':results['full']['foreground_saliency_iou']-results['frozen']['foreground_saliency_iou'],'cam_cosine_similarity':cosine,'cam_top_region_iou':overlap}); sample.update(results); examples.append(sample) if len(examples)<6 else None; processed+=1
        if processed>=max_samples: break
    for explainer in explainers.values(): explainer.close()
    pd.DataFrame(rows).to_csv(outdir/'gradcam_per_sample.csv',index=False); pd.DataFrame(paired).to_csv(outdir/'gradcam_paired.csv',index=False)
    if examples:
        fig,axes=plt.subplots(len(examples),3,figsize=(8,2.6*len(examples)),squeeze=False)
        for row,example in enumerate(examples):
            original=unnormalize(example['image']); axes[row,0].imshow(original); axes[row,0].set_title(f'Real: {class_names[example["label"]]}',fontsize=8)
            for col,strategy in enumerate(STRATEGIES,1): axes[row,col].imshow(original); axes[row,col].imshow(example[strategy]['cam'],cmap='jet',alpha=.45); axes[row,col].set_title(f'{strategy}: {class_names[example[strategy]["prediction"]]}',fontsize=8)
            for axis in axes[row]: axis.axis('off')
        fig.tight_layout(); fig.savefig(outdir/'gradcam_examples.png',dpi=220,bbox_inches='tight'); plt.close(fig)
    summary={}
    for strategy in STRATEGIES:
        subset=[r for r in rows if r['strategy']==strategy]
        for metric in ('foreground_energy','pointing_game','foreground_saliency_iou'):
            values=[r[metric] for r in subset]; summary[f'{strategy}_{metric}_mean']=float(np.mean(values)); summary[f'{strategy}_{metric}_std']=float(np.std(values,ddof=1)) if len(values)>1 else 0.0
    for metric in ('foreground_energy_delta','pointing_game_delta','foreground_saliency_iou_delta','cam_cosine_similarity','cam_top_region_iou'): summary[f'{metric}_mean']=float(np.mean([r[metric] for r in paired]))
    return summary


In [ ]:
def run_one_seed(seed,epochs=15,train_fraction=1.0,test_limit=0,gradcam_samples=200):
    set_seed(seed); outdir=RESULTS_DIR/f'seed_{seed}'; outdir.mkdir(parents=True,exist_ok=True); train_loader,val_loader,test_loader,class_names=make_loaders(seed,train_fraction,test_limit)
    models={}; classification=[]
    for strategy in STRATEGIES:
        set_seed(seed); model=build_model(strategy,len(class_names)); model=train(model,strategy,train_loader,val_loader,len(class_names),epochs,outdir); metrics=evaluate(model,test_loader,len(class_names)); metrics.update(strategy=strategy); classification.append(metrics); models[strategy]=model
    spatial=analyze_gradcam(models,test_loader,class_names,outdir,gradcam_samples); summary={'classification':classification,'gradcam':spatial}; (outdir/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8'); return summary

RUN_QUICK=False
if RUN_QUICK: display(pd.DataFrame(run_one_seed(42,epochs=1,train_fraction=.03,test_limit=64,gradcam_samples=8)['classification']))
else: print('RUN_QUICK=False: teste rápido não executado. Altere para True para validar o notebook.')

In [ ]:
RUN_FULL=False
if RUN_FULL:
    for seed in (42,123,2026): run_one_seed(seed,epochs=15,train_fraction=1.0,test_limit=0,gradcam_samples=200)
else: print('RUN_FULL=False: experimento completo não executado. Altere para True para gerar os resultados.')

In [ ]:
def aggregate_results():
    paths=sorted(RESULTS_DIR.glob('seed_*/summary.json'))
    if not paths: print('Nenhum resultado encontrado. Execute RUN_FULL=True.'); return
    summaries=[json.loads(path.read_text(encoding='utf-8')) for path in paths]; class_rows=[]
    for strategy in STRATEGIES:
        rows=[next(item for item in summary['classification'] if item['strategy']==strategy) for summary in summaries]; row={'strategy':strategy,'runs':len(rows)}
        for metric in ('accuracy','macro_f1','balanced_accuracy','ece_10_bins','loss'):
            values=[float(item[metric]) for item in rows]; row[f'{metric}_mean']=float(np.mean(values)); row[f'{metric}_std']=float(np.std(values,ddof=1)) if len(values)>1 else 0.0
        class_rows.append(row)
    gradcam_rows=[]
    for metric in summaries[0]['gradcam']:
        values=[float(summary['gradcam'][metric]) for summary in summaries]; gradcam_rows.append({'metric':metric,'mean':float(np.mean(values)),'std':float(np.std(values,ddof=1)) if len(values)>1 else 0.0,'runs':len(values)})
    aggregate_dir=RESULTS_DIR/'aggregate'; aggregate_dir.mkdir(exist_ok=True); pd.DataFrame(class_rows).to_csv(aggregate_dir/'classification_aggregate.csv',index=False); pd.DataFrame(gradcam_rows).to_csv(aggregate_dir/'gradcam_aggregate.csv',index=False); display(pd.DataFrame(class_rows)); display(pd.DataFrame(gradcam_rows))

if RUN_FULL: aggregate_results()
else: print('Agregação ignorada porque RUN_FULL=False. Mantenha RUN_FULL=True após o experimento completo.')

In [ ]:
# Comparação direta entre frozen e full
aggregate_dir = RESULTS_DIR / 'aggregate'
classification_path = aggregate_dir / 'classification_aggregate.csv'
gradcam_path = aggregate_dir / 'gradcam_aggregate.csv'
if not classification_path.exists() or not gradcam_path.exists():
    print('Resultados agregados não encontrados. Execute RUN_FULL=True e a célula de agregação antes desta.')
else:
    classification = pd.read_csv(classification_path).set_index('strategy')
    gradcam = pd.read_csv(gradcam_path).set_index('metric')['mean']
    comparison = pd.DataFrame({
        'frozen': [classification.loc['frozen', 'accuracy_mean'], gradcam['frozen_foreground_energy_mean'], gradcam['frozen_pointing_game_mean'], gradcam['frozen_foreground_saliency_iou_mean']],
        'full': [classification.loc['full', 'accuracy_mean'], gradcam['full_foreground_energy_mean'], gradcam['full_pointing_game_mean'], gradcam['full_foreground_saliency_iou_mean']],
    }, index=['acurácia', 'foreground_energy', 'pointing_game', 'foreground_saliency_iou'])
    display(comparison.style.format('{:.4f}'))
    accuracy_delta = comparison.loc['acurácia', 'full'] - comparison.loc['acurácia', 'frozen']
    energy_delta = comparison.loc['foreground_energy', 'full'] - comparison.loc['foreground_energy', 'frozen']
    pointing_delta = comparison.loc['pointing_game', 'full'] - comparison.loc['pointing_game', 'frozen']
    iou_delta = comparison.loc['foreground_saliency_iou', 'full'] - comparison.loc['foreground_saliency_iou', 'frozen']
    print(f'Variações full − frozen: acurácia={accuracy_delta:+.4f}; foreground_energy={energy_delta:+.4f}; pointing_game={pointing_delta:+.4f}; foreground_saliency_iou={iou_delta:+.4f}')
    if accuracy_delta > 0 and energy_delta < 0 and pointing_delta < 0 and iou_delta < 0:
        print('Interpretação: a acurácia aumentou, mas os três indicadores espaciais pioraram. Isso é evidência compatível com maior uso do fundo ou de correlações espúrias; confirme com ablação/oclusão do fundo.')
    elif accuracy_delta > 0 and energy_delta > 0 and pointing_delta > 0 and iou_delta > 0:
        print('Interpretação: o fine-tuning melhorou a acurácia e o alinhamento das ativações com o objeto. Esse é o cenário mais favorável.')
    else:
        print('Interpretação: o resultado é misto. Descreva cada métrica separadamente e evite concluir que houve atalho pelo fundo sem um teste adicional.')

## Interpretação e ordem de execução

1. Para validar a instalação, defina `RUN_QUICK=True` na célula correspondente e execute-a.
2. Depois volte `RUN_QUICK=False`, defina `RUN_FULL=True` e execute a célula do experimento; aguarde os três seeds terminarem.
3. Ainda com `RUN_FULL=True`, execute a célula de agregação.
4. Execute a célula de comparação: ela exibirá a tabela e uma interpretação automática.

Compare `foreground_energy`, `pointing_game` e `foreground_saliency_iou` entre `frozen` e `full`. Um aumento de acurácia acompanhado de pior alinhamento espacial pode indicar que o fine-tuning passou a explorar o fundo ou outras correlações espúrias.